In [20]:
#import all the library
import asyncio
import random
import time

In [3]:
async def call_tool(name: str, delay: float, should_fail: bool = False) -> str:
    """Simulates calling an external tool/API/model"""
    await asyncio.sleep(delay) #this is called the corutinee
    if should_fail:
        raise ValueError(f"{name} failed to respond")
    return f"{name} → success"


In [17]:
# Pattern 1: asyncio.gather() — basic concurrency

async def demo_gather():
    print("\n--- Pattern 1: Basic Concurrent Calls ---")
    start = time.time()

    results = await asyncio.gather(
        call_tool("Translation", 1),
        call_tool("SymptomExtraction", 1),
        call_tool("WebSearch", 1)
    )

    print(results)
    print(f"Time: {time.time() - start:.2f}s (should be ~1s, not 3s)")


In [13]:
async def demo_partial_failure():
    print("\n--- Partial Failure Handling ---")

    results = await asyncio.gather(
        call_tool("Translation", 1),
        call_tool("DrugLookup", 1, should_fail=True),
        call_tool("BedAvailability", 1),
        return_exceptions=True
    )

    for r in results:
        if isinstance(r, Exception):
            print(f"  Failed: {r}")
        else:
            print(f"  OK: {r}")

In [19]:
async def demo_timeout():
    print("\n--- Pattern 3: Timeout Handling ---")

    try:
        result = await asyncio.wait_for(
            call_tool("SlowModel", delay=5),  # takes 5s
            timeout=2.0                        # but we only wait 2s
        )
        print(result)
    except asyncio.TimeoutError:
        print("  TimeoutError caught — using fallback response")

In [21]:

async def demo_semaphore():
    print("\n--- Pattern 4: Rate Limiting with Semaphore ---")

    semaphore = asyncio.Semaphore(2)  # max 2 concurrent calls
    async def limited_call(name: str, delay: float):
        async with semaphore:
            print(f"  {name} started")
            await asyncio.sleep(delay)
            print(f"  {name} finished")

    start = time.time()
    await asyncio.gather(
        limited_call("Request-1", 1),
        limited_call("Request-2", 1),
        limited_call("Request-3", 1),
        limited_call("Request-4", 1),
    )
    print(f"Time: {time.time() - start:.2f}s (should be ~2s, not 1s — only 2 run at once)")


In [27]:
async def stream_tokens(text: str):
    """Simulates a model streaming output token by token"""
    for word in text.split():
        await asyncio.sleep(0.90)
        yield word

async def demo_streaming():
    print("\n--- Pattern 5: Streaming with Async Generator ---")
    async for token in stream_tokens("Patient needs urgent care now"):
        print(token, end=" ", flush=True)
    print()


In [31]:
async def background_log(message: str):
    await asyncio.sleep(1)
    print(f"  [background log written]: {message}")

async def demo_create_task():
    print("\n--- Pattern 6: Fire-and-Forget Background Task ---")

    task = asyncio.create_task(background_log("Session completed"))

    print("  Main work continuing immediately...")
    await asyncio.sleep(0.2)
    print("  Main work done, now waiting for background task")

    await task

In [36]:
async def main():
    await demo_gather()
    await demo_partial_failure()
    await demo_timeout()
    await demo_semaphore()
    await demo_streaming()
    await demo_create_task()

await main()

/usr/lib/python3.12/codeop.py:126: RuntimeWarning: coroutine 'main' was never awaited
  codeob = compile(source, filename, symbol, flags, True)



--- Pattern 1: Basic Concurrent Calls ---
['Translation → success', 'SymptomExtraction → success', 'WebSearch → success']
Time: 1.00s (should be ~1s, not 3s)

--- Partial Failure Handling ---
  OK: Translation → success
  Failed: DrugLookup failed to respond
  OK: BedAvailability → success

--- Pattern 3: Timeout Handling ---
  TimeoutError caught — using fallback response

--- Pattern 4: Rate Limiting with Semaphore ---
  Request-1 started
  Request-2 started
  Request-1 finished
  Request-2 finished
  Request-3 started
  Request-4 started
  Request-3 finished
  Request-4 finished
Time: 2.00s (should be ~2s, not 1s — only 2 run at once)

--- Pattern 5: Streaming with Async Generator ---
Patient needs urgent care now 

--- Pattern 6: Fire-and-Forget Background Task ---
  Main work continuing immediately...
  Main work done, now waiting for background task
  [background log written]: Session completed
